# DEST — CIFAR100 collatz_v3 (6 seeds finales)
Solo las 6 semillas que faltan (46-51). Ejecuta las 3 celdas en orden.

In [ ]:
import os, sys, json, zipfile, torch, subprocess, importlib

# 1. Instalar dependencias opcionales que Colab no trae
try:
    import seaborn, sklearn, tqdm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'seaborn', 'scikit-learn', 'tqdm', 'matplotlib', '-q'])

# 2. Escribir dest_lib en disco
lib_files = {
    "__init__.py": "# dest_lib package\n"
}

os.makedirs('dest_lib', exist_ok=True)
for filename, content in lib_files.items():
    fpath = os.path.join('dest_lib', filename)
    with open(fpath, 'w', encoding='utf-8') as fh:
        fh.write(content)

# 3. Verificar que los archivos existen
created = sorted(os.listdir('dest_lib'))
print('Archivos en dest_lib:', created)
assert 'config.py' in created, 'ERROR: config.py no fue creado!'
assert 'runner.py' in created, 'ERROR: runner.py no fue creado!'

# 4. Limpiar cache de Python para que detecte los archivos nuevos
importlib.invalidate_caches()
for mod_name in list(sys.modules.keys()):
    if 'dest_lib' in mod_name:
        del sys.modules[mod_name]

# 5. Agregar directorio actual al path e importar
cwd = os.path.abspath('.')
if cwd not in sys.path:
    sys.path.insert(0, cwd)

from dest_lib.config import get_config
from dest_lib.runner import ExperimentRunner
print('Imports OK')

CONFIG = get_config('PAPER')
CONFIG['output_dir'] = os.path.join(cwd, 'dest_results_paper')
os.makedirs(CONFIG['output_dir'], exist_ok=True)
runner = ExperimentRunner(CONFIG)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
FALTANTES = [
    [
        "CIFAR100",
        "collatz_v3",
        46
    ],
    [
        "CIFAR100",
        "collatz_v3",
        47
    ],
    [
        "CIFAR100",
        "collatz_v3",
        48
    ],
    [
        "CIFAR100",
        "collatz_v3",
        49
    ],
    [
        "CIFAR100",
        "collatz_v3",
        50
    ],
    [
        "CIFAR100",
        "collatz_v3",
        51
    ]
]

print(f'Semillas a entrenar: {len(FALTANTES)} (collatz_v3 seeds 46-51)')

for ds, sampler, seed in FALTANTES:
    print('\n' + '='*60)
    print(f'DS={ds} | SAMPLER={sampler} | SEED={seed}')
    print('='*60)
    res = runner.run_single_seed(
        exp_id=f'{ds}_{sampler}',
        sampler_name=sampler,
        seed=seed,
        dataset=ds,
        dropout_mode='deterministic',
    )
    print(f'Semilla {seed} completada. Acc: {res.final_test_acc:.2f}%')

In [ ]:
zip_filename = 'resultados_CIFAR100_Final.zip'
print(f'Empaquetando en {zip_filename}...')
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(CONFIG['output_dir']):
        for file in files:
            if not file.endswith('.zip'):
                fpath = os.path.join(root, file)
                arcname = os.path.relpath(fpath, CONFIG['output_dir'])
                zipf.write(fpath, arcname)
size_mb = os.path.getsize(zip_filename) / 1024 / 1024
print(f'ZIP listo: {zip_filename} ({size_mb:.2f} MB)')
try:
    from google.colab import files
    files.download(zip_filename)
    print('Descarga iniciada.')
except ImportError:
    print(f'Descarga manual: {zip_filename}')